In [12]:
import pandas as pd
import numpy as np

Download a publicly available CSV dataset of IPL cricket matches (for example, from Kaggle or data.world), and use pandas to load it into a DataFrame named ipl_df. Display the first 5 rows.

In [13]:
ipl_df = pd.read_csv("C:/Users/Windows/OneDrive/Desktop/Pandas Dataset/matches.csv")
ipl_df.head(5)

,id,season,city,date,team1,team2,toss_winner,toss_decision,result,dl_applied,winner,win_by_runs,win_by_wickets,player_of_match,venue,umpire1,umpire2,umpire3
0,1,2017,Hyderabad,2017-04-05,Sunrisers Hyderabad,Royal Challengers Bangalore,Royal Challengers Bangalore,field,normal,0,Sunrisers Hyderabad,35,0,Yuvraj Singh,"Rajiv Gandhi International Stadium, Uppal",AY Dandekar,NJ Llong,NaN
1,2,2017,Pune,2017-04-06,Mumbai Indians,Rising Pune Supergiant,Rising Pune Supergiant,field,normal,0,Rising Pune Supergiant,0,7,SPD Smith,Maharashtra Cricket Association Stadium,A Nand Kishore,S Ravi,NaN
2,3,2017,Rajkot,2017-04-07,Gujarat Lions,Kolkata Knight Riders,Kolkata Knight Riders,field,normal,0,Kolkata Knight Riders,0,10,CA Lynn,Saurashtra Cricket Association Stadium,Nitin Menon,CK Nandan,NaN
3,4,2017,Indore,2017-04-08,Rising Pune Supergiant,Kings XI Punjab,Kings XI Punjab,field,normal,0,Kings XI Punjab,0,6,GJ Maxwell,Holkar Cricket Stadium,AK Chaudhary,C Shamshuddin,NaN
4,5,2017,Bangalore,2017-04-08,Royal Challengers Bangalore,Delhi Daredevils,Royal Challengers Bangalore,bat,normal,0,Royal Challengers Bangalore,15,0,KM Jadhav,M Chinnaswamy Stadium,NaN,NaN,NaN


Clean the ipl_df DataFrame by removing any rows where the 'winner' column is missing or null, and reset the index afterwards.

In [14]:
ipl_df = ipl_df.dropna(subset=['winner']).reset_index(drop=True)
print(ipl_df.shape)
print(ipl_df['winner'].isnull().sum())

(752, 18)
0


Transform the ipl_df DataFrame to add a new column 'match_margin_type' that labels each match as 'Runs' if the 'win_by_runs' value is greater than 0, 'Wickets' if 'win_by_wickets' is greater than 0, or 'Tie/No Result' otherwise.<br><br><em><strong>Hint:</strong> Use numpy.select() or pandas' apply() for this transformation.</em>

In [15]:
conditions = [
    ipl_df["win_by_runs"] > 0,
    ipl_df["win_by_wickets"] > 0,
]
choices = ["Runs", "Wickets"]

ipl_df["match_margin_type"] = np.select(conditions, choices, default="Tie/No Result")

print(ipl_df["match_margin_type"].value_counts())

match_margin_type
Wickets          406
Runs             337
Tie/No Result      9
Name: count, dtype: int64



Merge the ipl_df DataFrame with a separate DataFrame containing IPL team names and their home cities (create this manually with at least 5 teams), using the team name as the key, and add the home city for both 'team1' and 'team2' in each match.

In [17]:
teams_df = pd.DataFrame({
    'team_name': [
        'Gujarat Titans', 
        'Chennai Super Kings', 
        'Mumbai Indians', 
        'Lucknow Super Giants', 
        'Royal Challengers Bangalore'
    ],
    'home_city': [
        'Ahmedabad', 
        'Chennai', 
        'Mumbai', 
        'Lucknow', 
        'Bengaluru'
    ]
})


ipl_df = ipl_df.merge(
    teams_df, 
    left_on='team1', 
    right_on='team_name', 
    how='left'
).rename(columns={'home_city': 'team1_home_city'}).drop(columns=['team_name'])


ipl_df = ipl_df.merge(
    teams_df, 
    left_on='team2', 
    right_on='team_name', 
    how='left'
).rename(columns={'home_city': 'team2_home_city'}).drop(columns=['team_name'])


print(ipl_df[['id', 'team1', 'team1_home_city', 'team2', 'team2_home_city']])

        id                        team1 team1_home_city team1_home_city  \
0        1          Sunrisers Hyderabad       Hyderabad             NaN   
1        2               Mumbai Indians          Mumbai          Mumbai   
2        3                Gujarat Lions          Rajkot             NaN   
3        4       Rising Pune Supergiant            Pune             NaN   
4        5  Royal Challengers Bangalore       Bangalore       Bengaluru   
..     ...                          ...             ...             ...   
747  11347        Kolkata Knight Riders         Kolkata             NaN   
748  11412          Chennai Super Kings         Chennai         Chennai   
749  11413          Sunrisers Hyderabad       Hyderabad             NaN   
750  11414               Delhi Capitals           Delhi             NaN   
751  11415               Mumbai Indians          Mumbai          Mumbai   

                           team2 team2_home_city team2_home_city  
0    Royal Challengers Bangalore

Create a pivot table to show the total number of matches won by each team per season, and display the top 3 teams with the most wins in any single season.

In [20]:
pivot_wins = ipl_df.pivot_table(
    index='winner', 
    columns='season', 
    values='id', 
    aggfunc='count', 
    fill_value=0
)

top_3_season_wins = (
    ipl_df.groupby(['season', 'winner'])
    .size()
    .reset_index(name='wins')
    .sort_values(by='wins', ascending=False)
    .head(3)
    .reset_index(drop=True)
)

print("--- Pivot Table: Wins per Season ---")
print(pivot_wins)

print("\n--- Top 3 Teams with Most Wins in a Single Season ---")
print(top_3_season_wins)

--- Pivot Table: Wins per Season ---
season                       2008  2009  2010  2011  2012  2013  2014  2015  \
winner                                                                        
Chennai Super Kings             9     8     9    11    10    12    10    10   
Deccan Chargers                 2     9     8     6     4     0     0     0   
Delhi Capitals                  0     0     0     0     0     0     0     0   
Delhi Daredevils                7    10     7     4    11     3     2     5   
Gujarat Lions                   0     0     0     0     0     0     0     0   
Kings XI Punjab                10     7     4     7     8     8    12     3   
Kochi Tuskers Kerala            0     0     0     6     0     0     0     0   
Kolkata Knight Riders           6     3     7     8    12     6    11     7   
Mumbai Indians                  7     5    11    10    10    13     7    10   
Pune Warriors                   0     0     0     4     4     4     0     0   
Rajasthan Royal